In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from google.colab import files
from nltk.stem import SnowballStemmer
from wordcloud import WordCloud
from sklearn.utils import resample
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

# Data preprocessing for clasification model

In [ ]:
# Install only the core missing components
!pip install -q sentence-transformers
!pip install -q plotly accelerate

print("Environment cleaned and updated.")

Environment cleaned and updated.


In [ ]:
from google.colab import files
import os

# Upload kaggle.json when prompted
files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Download the dataset
!kaggle datasets download -d datafiniti/consumer-reviews-of-amazon-products
!unzip -q consumer-reviews-of-amazon-products.zip -d data/
!ls data/

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load — pick whichever CSV file appeared in data/
df = pd.read_csv("data/1429_1.csv")

print(df.shape)
df.head(3)

In [ ]:
# ── What columns do we have? ──────────────────────────────────────────
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())

In [ ]:
# ── Rating distribution ───────────────────────────────────────────────
sns.countplot(x="reviews.rating", data=df, palette="viridis")
plt.title("Star Rating Distribution")
plt.show()

print(df["reviews.rating"].value_counts())

In [ ]:
def map_sentiment(rating):
    if rating <= 2:
        return 0   # Negative
    elif rating == 3:
        return 1   # Neutral
    else:
        return 2   # Positive

df["label"] = df["reviews.rating"].apply(map_sentiment)

# Check class balance
print(df["label"].value_counts())
sns.countplot(x="label", data=df, palette="Set2")
plt.xticks([0,1,2], ["Negative", "Neutral", "Positive"])
plt.title("Sentiment Class Distribution")
plt.show()

In [ ]:
# 1. Select only required columns
df = df[['reviews.text', 'reviews.title', 'reviews.rating']]

# 2. Drop rows where any of these three columns have NaN values
# This removes the 1 missing text, 33 missing ratings, and 6 missing titles
df = df.dropna(subset=['reviews.text', 'reviews.title', 'reviews.rating'])

# 3. Final check
print(f"Current shape: {df.shape}")
print("\nMissing values check:")
print(df.isnull().sum())

In [ ]:
# 1. Combine title and text into one column
# We add a space between them so the last word of the title
# and the first word of the text don't merge.
df['full_review'] = df['reviews.title'] + " " + df['reviews.text']

# 2. Ensure rating is an integer (optional but cleaner for classification)
df['reviews.rating'] = df['reviews.rating'].astype(int)

# 3. Drop the old columns as we now have 'full_review'
df = df[['full_review', 'reviews.rating']]

# 4. Preview the result
print(df.info())
print("\nFirst 5 rows:")
print(df.head())

In [ ]:
# Print 3 examples for each rating (1 to 5)
for rating in range(1, 6):
    print(f"==================== RATING: {rating} ====================")
    examples = df[df['reviews.rating'] == rating]['full_review'].head(3)
    for i, ex in enumerate(examples, 1):
        print(f"Example {i}: {ex[:200]}...") # Printing first 200 chars for brevity
    print("\n")

In [ ]:
from nltk.stem import SnowballStemmer

# Initialize the Snowball Stemmer for English
stemmer = SnowballStemmer("english")

def preprocess_with_stemming(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Tokenize and Stem (keeping punctuation as separate tokens if they are attached)
    # We split by space to keep the punctuation attached to words or as separate entities
    words = text.split()
    stemmed_words = [stemmer.stem(w) for w in words]

    return " ".join(stemmed_words)

# Apply this new preprocessing
df['cleaned_review'] = df['full_review'].apply(preprocess_with_stemming)

# Preview the results to see how stemming affected the words and punctuation
print("Stemming results (with punctuation):")
print(df[['full_review', 'cleaned_review']].head())

Checking the initial distribution of the dataset to identify which rating category has the fewest reviews, which dictates the size of our balanced dataset.

Misklassification detection

In [ ]:
# Check the raw counts of each rating in the original cleaned dataframe
raw_counts = df['reviews.rating'].value_counts().sort_index()

print("--- Original Dataset Distribution ---")
print(raw_counts)

# Identify the absolute minimum
min_rating = raw_counts.idxmin()
min_value = raw_counts.min()

print(f"\nThe smallest class is Rating {min_rating} with only {min_value} reviews.")

Converting the 5-star rating system into 3 Sentiment Categories (0: Negative, 1: Neutral, 2: Positive) to improve model stability and increase the usable data size for underrepresented classes.

In [ ]:
# 1. Use the dataframe BEFORE previous downsampling to get more data
# Assuming 'df' is your cleaned dataframe with all 34k reviews
def group_ratings(rating):
    if rating <= 2: return 0  # Negative (1 & 2 stars)
    elif rating == 3: return 1 # Neutral (3 stars)
    else: return 2            # Positive (4 & 5 stars)

df['sentiment'] = df['reviews.rating'].apply(group_ratings)

# 2. Calculate new group sizes
group_counts = df['sentiment'].value_counts()
# The bottleneck is now the smaller of Negative (1367) or Neutral (1499)
target_size = group_counts.min()

# 3. Perform Down-sampling
from sklearn.utils import resample
balanced_list = []
for sentiment in [0, 1, 2]:
    new_sample = resample(df[df['sentiment'] == sentiment],
                         replace=False,
                         n_samples=target_size,
                         random_state=42)
    balanced_list.append(new_sample)

df_balanced = pd.concat(balanced_list)

print("--- New Maximum Balance ---")
print(df_balanced['sentiment'].value_counts())
print(f"\nTotal rows now: {len(df_balanced)}")

Generating **Word Clouds** for Negative and Positive reviews to see if our **Stemming** and **Negation Handling** (e.g., "NOT_") are visible and making sense before vectorization.

In [ ]:
from wordcloud import WordCloud
# 1. Define the negation handling function again just in case
def handle_negation(text):
    if not isinstance(text, str):
        return ""
    negations = ["not", "no", "never", "isnt", "wasnt", "arent", "werent", "dont", "doesnt", "didnt", "havent", "hasnt", "hadnt", "cant", "couldnt", "shouldnt", "wont", "wouldnt"]
    words = text.split()
    for i in range(len(words) - 1):
        if words[i] in negations:
            words[i+1] = "NOT_" + words[i+1]
    return " ".join(words)

# 2. Apply it to the balanced dataframe to create the column
# We use 'cleaned_review' which was created during the stemming step
df_balanced['advanced_review'] = df_balanced['cleaned_review'].apply(handle_negation)

# 3. Verify the column exists now
print("Columns in df_balanced:", df_balanced.columns.tolist())
print(df_balanced[['advanced_review']].head())

def plot_wordcloud(sentiment_label, title):
    # Combine all reviews for a specific sentiment
    text = " ".join(review for review in df_balanced[df_balanced['sentiment'] == sentiment_label]['advanced_review'])

    wordcloud = WordCloud(width=800, height=400,
                          background_color='white',
                          max_words=100,
                          colormap='magma').generate(text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(title, fontsize=15)
    plt.axis('off')
    plt.show()

# Plot for Negative (0) and Positive (2)
plot_wordcloud(0, "Most Frequent Words in Negative Reviews")
plot_wordcloud(1, "Most Frequent Words in Neutral Reviews")
plot_wordcloud(2, "Most Frequent Words in Positive Reviews")

Implementing **Advanced Negation Handling** to treat negations and their following words as a single semantic unit (e.g., "not_good"). This prevents the model from misinterpreting "not good" as a positive sentiment due to the word "good".

In [ ]:
# List of common English negations
negations_list = ["not", "no", "never", "isnt", "wasnt", "arent", "werent",
                  "dont", "doesnt", "didnt", "havent", "hasnt", "hadnt",
                  "cant", "couldnt", "shouldnt", "wont", "wouldnt"]

def process_negations(text):
    if not isinstance(text, str):
        return ""

    words = text.split()
    transformed_words = []
    i = 0

    while i < len(words):
        word = words[i]
        # If the word is a negation and not the last word in the text
        if word in negations_list and i + 1 < len(words):
            # Combine the negation with the next word using an underscore
            next_word = words[i+1]
            transformed_words.append(f"{word}_{next_word}")
            i += 2  # Skip the next word as it's now combined
        else:
            transformed_words.append(word)
            i += 1

    return " ".join(transformed_words)

# Apply the advanced transformation to our balanced dataset
df_balanced['advanced_review'] = df_balanced['cleaned_review'].apply(process_negations)

# Show an example where negation was handled
example_index = df_balanced[df_balanced['cleaned_review'].str.contains('not ')].index[0]
print("Original Stemmed Text:", df_balanced.loc[example_index, 'cleaned_review'])
print("Advanced Negation Text:", df_balanced.loc[example_index, 'advanced_review'])

Visualizing how the text looks now that it's prepared for **N-grams**. This allows us to see common word pairs (bigrams) that the model will use to distinguish between sentiments.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def plot_top_ngrams(text_series, n=2, top_k=10, title="Top Bigrams"):
    vec = CountVectorizer(ngram_range=(n, n)).fit(text_series)
    bag_of_words = vec.transform(text_series)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)

    ngram_df = pd.DataFrame(words_freq[:top_k], columns=['Ngram', 'Frequency'])

    plt.figure(figsize=(10, 6))
    sns.barplot(x='Frequency', y='Ngram', data=ngram_df, palette='rocket')
    plt.title(title)
    plt.show()

# Plot Top 2-grams (Bigrams) for Negative reviews to see if "not_something" appears
plot_top_ngrams(df_balanced[df_balanced['sentiment'] == 0]['advanced_review'], n=2, title="Top Negative Bigrams")

Converting the processed text into numerical features using **TF-IDF**. We set `ngram_range=(1, 2)` to capture both individual words and word pairs (like "not_work"), and `max_features=5000` to keep the most informative terms.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF
# ngram_range=(1, 2) includes both single words and bigrams
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)

# Create the feature matrix X and the target y
X = tfidf.fit_transform(df_balanced['advanced_review'])
y = df_balanced['sentiment']

print(f"Feature Matrix Shape: {X.shape}")
print("TF-IDF extraction with bigrams complete.")

### Data Preprocessing summary

*   **Data Loading & Initial Check:** The Amazon product reviews dataset was loaded and examined, revealing 34,660 initial entries and identifying missing values in titles, text, and ratings.
*   **Rating Distribution:** Exploring the ratings showed a heavy imbalance, with 5-star reviews (23,775) vastly outnumbering 1-star (410) and 2-star (402) reviews.
*   **Data Cleaning:** Rows with missing values in critical columns (`reviews.text`, `reviews.title`, `reviews.rating`) were dropped, resulting in a cleaner dataset of 34,620 reviews.
*   **Feature Engineering (Text & Ratings):** The review title and text were combined into a single `full_review` feature, and the 5-star ratings were mapped to 3 sentiment classes (0: Negative, 1: Neutral, 2: Positive).
*   **Text Normalization (Stemming):** A Snowball Stemmer was applied to reduce words to their root forms (e.g., 'disappointed' to 'disappoint'), keeping punctuation intact for now.
*   **Class Balancing:** To handle the severe class imbalance and improve model stability, the dataset was downsampled based on the minority class (Negative), resulting in a perfectly balanced dataset of 2,436 reviews (812 per sentiment).
*   **Negation Handling:** An advanced negation handling function was implemented to fuse negation words with the subsequent word (e.g., 'not great' becomes 'not_great'), preventing misinterpretation of sentiment.
*   **N-gram Analysis:** Visualizing top bigrams (2-grams) helped confirm that the negation handling was working and highlighted common word pairs associated with different sentiments.
*   **TF-IDF Vectorization:** The processed text was converted into a numerical feature matrix using TF-IDF (Term Frequency-Inverse Document Frequency) with bigrams, creating a 5,000-dimensional space ready for modeling.